# Phụ lục A. Tái lập và cập nhật báo cáo

## A.1 Hai mức tái lập

**Tái lập Book công khai:** clone repository báo cáo, cài requirements và build. Các notebook
đọc `report_summary.json`, sinh bảng/biểu đồ và chạy ví dụ tổng hợp. Không cần API key,
quyền Drive hoặc corpus gốc để xem lại phần trình bày.

**Tái lập xử lý từ nguồn:** cần snapshot M1 riêng của nhóm, code parser và cấu hình root.
Chạy source audit, export M1, parser và validation để thu kết quả mới; sau đó mới tổng hợp
số liệu công khai. Lệnh dưới đây là template, cần thay các đường dẫn bằng input thật.

```bash
python -m pip install -e ".[report]"
python -m src.crawler.validate_sources --publications INPUT/publications.parquet --raw-dir INPUT/raw --output OUTPUT/source_audit.parquet
python -m src.parser.main --publications INPUT/publications.parquet --raw-dir INPUT/raw --output-dir OUTPUT/parsed --root-overrides configs/submission_parser_roots_20261106.json
python -m src.parser.validation --course-dir INPUT/course --parsed-dir OUTPUT/parsed --publications INPUT/publications.parquet --output OUTPUT/parsed/project_validation.json
```

`--resume` chỉ dùng khi tiếp tục cùng output và phạm vi hợp lệ. Khi thay input, root hoặc
source, fingerprint giúp xác định publication cần xử lý lại. Không dùng output cũ mà
ghép vào manifest của một run khác.

## A.2 Môi trường dùng cho phép đo và môi trường compile

Các số đo lịch sử giữ Python/thư viện của chính run đó. Môi trường compile Book được pin
riêng trong requirements, không làm thay đổi ý nghĩa của số đo đã ghi trước đây.
Các ví dụ thuật toán của báo cáo import code từ commit ghim, không dùng dữ liệu nghiên cứu thật.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2 = data['scope'], data['m1'], data['m2']
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})
print(f"Snapshot: {scope['papers']} paper / {scope['versions']} version; báo cáo ngày {data['report_date']}")

In [ ]:
import sys, importlib.metadata
print('Python compile Book:',sys.version.split()[0])
packages=['jupyter-book','pandas','pyarrow','matplotlib','nbformat','arxiv-harvester']
display(pd.DataFrame([{'package':name,'installed_version':importlib.metadata.version(name)} for name in packages]))

## A.3 Build Book và deploy

```bash
git clone https://github.com/NguyenQuocDat06-AI/arxiv-harvester-book.git
cd arxiv-harvester-book
python -m pip install -r requirements.txt
jupyter-book build . --all -W
```

Sửa `index.md`, notebook trong `chapters/` hoặc nội dung tổng hợp đã xác minh. Sau khi build
đạt, commit và push `main` của repository Book. Workflow thực thi lại notebook rồi deploy
Pages; không dùng ảnh chụp màn hình làm nguồn cho các bảng thống kê.

URL nộp cố định:

**https://nguyenquocdat06-ai.github.io/arxiv-harvester-book/**

Giữ nguyên tên tài khoản, repository và Pages. Code M1/M2 tiếp tục nằm ở repository code
chung, nhánh `develop`; không cần đổi tên repository Book khi bổ sung dữ liệu.

## A.4 Những gì giữ ngoài GitHub

Corpus TeX/Bib, Parquet làm việc, manifest gốc, log đầy đủ, bảng mã paper và credential
được giữ ở nơi làm việc của nhóm. Book công khai chỉ có thống kê tổng hợp, nhãn coverage
và ví dụ tổng hợp. Test được giữ local theo cách tổ chức hiện tại của nhóm.

## A.5 Thao tác nộp

Mở Book không đăng nhập, kiểm tra hai phần M1/M2, dán link Book và link source vào Moodle,
lưu bài rồi kiểm tra trạng thái nộp. Việc có Pages không tự thực hiện bước nộp trên Moodle.

## A.6 Tổng hợp log benchmark theo phiên

Dùng cùng input/phạm vi đã chốt và ghi code/config của từng phiên Colab. Giữ nguyên
`stats.json`, `run_metadata.json` và resource samples của từng run; đưa từng bản stats
vào lệnh dưới đây đúng một lần. File metadata, nếu có, nằm cạnh stats và phải trùng run ID.

```bash
python -m src.crawler.benchmark_sessions --stats LOGS/run1/stats.json LOGS/run2/stats.json --final-output-dir OUTPUT_FINAL --output benchmark_m1_sessions.json
```

Với M2, dùng các stats của parser trong một lệnh riêng. Output phân biệt profiler time,
child wall, số lượt xử lý, cache, RAM/đĩa và môi trường. Thiếu child wall ở một phiên làm
tổng child wall giữ null; không dùng profiler time của phiên đó để điền vào cùng cột.

Đối chiếu ID và version trên output hợp nhất bằng source audit và project validation.
Không dùng tổng counter theo phiên để suy ra số paper duy nhất. Khác phần cứng hoặc
code giữa phiên cần được công bố khi phân tích hiệu năng.

## Tài liệu và mã tham chiếu

- [Đặc tả M2](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API: cấu trúc truy vấn và Atom](https://info.arxiv.org/help/api/user-manual.html).
- [Quy định truy cập arXiv](https://info.arxiv.org/help/api/tou.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã triển khai: [tools/build_public_submission_book.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/build_public_submission_book.py).
- Mã triển khai: [tools/prepare_submission_snapshot.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/prepare_submission_snapshot.py).
- Mã triển khai: [tools/prepare_parser_colab.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/prepare_parser_colab.py).